# Hybrid quantum-classical portfolio optimizer: benchmark walkthrough

Simulation only: no QPU is used anywhere in this notebook.

Part 1 verifies the QUBO on a tiny instance from scratch (runs in seconds).
Part 2 loads the stored benchmark produced by

```bash
python -m hqpo.prepare_data --config configs/demo.yaml
python -m hqpo.run --config configs/demo.yaml --solver all
python -m hqpo.benchmark --config configs/demo.yaml
```

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from hqpo.data.synthetic import planted_instance, random_instance
from hqpo.models import Constraints, build_qubo
from hqpo.solvers import Budget, get_solver
from hqpo.solvers.classical import brute_force_optimum

ROOT = Path.cwd() if (Path.cwd() / "configs").exists() else Path.cwd().parent
EXPERIMENT = "demo"

## 1. Verify the encoding on a tiny instance

Six assets, 25 % lots, 50 % position limit, at most three names.

In [ ]:
problem = random_instance(6, Constraints(lots=4, max_weight=0.5, cardinality=3), seed=101)
qubo = build_qubo(problem)
print("qubits:", qubo.qubit_breakdown(), " penalty:", round(qubo.penalty, 4))

z_star, f_star, n_states = brute_force_optimum(problem)
energies = qubo.energies_all()                       # all 2^18 bitstrings
z_qubo = qubo.decode(qubo.index_to_bits(int(np.argmin(energies))))
print(f"brute force over {n_states} lot vectors: f* = {f_star:.6f}, weights = {problem.weights(z_star)}")
print(f"QUBO ground state:                  E* = {energies.min():.6f}, weights = {problem.weights(z_qubo)}")
assert problem.is_feasible(z_qubo) and abs(energies.min() - f_star) < 1e-9

## 2. Same problem, same budget, every solver

In [ ]:
budget = Budget(wall_time_s=3.0, max_evals=5_000_000)
rows = []
for name in ["cvxpy-relax", "miqp", "cpsat", "greedy", "sa", "qubo-sa", "random-qubo", "qaoa"]:
    r = get_solver(name).solve(problem, budget, seed=11)
    rows.append({"solver": name, "status": r.status, "gap": r.objective - f_star,
                 "raw violation": r.raw_violation, "repaired": r.repaired, "evals": r.evals,
                 "wall s": round(r.wall_time_s, 3), "qubits": r.meta.get("n_qubits"),
                 "depth": r.meta.get("depth"),
                 "feasible shots": r.meta.get("sample_feasible_fraction"),
                 "P(ground state)": r.meta.get("p_ground_state")})
pd.DataFrame(rows).set_index("solver")

`random-qubo` is the control: it draws the same number of uniformly random bitstrings and keeps the best.
Where QAOA only matches it, the result comes from sampling many candidates, not from the circuit.

## 3. An instance with an analytically known optimum

In [ ]:
planted, z_known = planted_instance(5, Constraints(lots=4, max_weight=0.5, cardinality=3), seed=2)
for name in ["miqp", "greedy", "qaoa"]:
    r = get_solver(name).solve(planted, budget, seed=11)
    print(f"{name:8s} gap to known optimum = {r.objective - planted.objective(z_known):+.2e}")

## 4. Stored benchmark results

Raw rows: one per (instance, solver, seed).

In [ ]:
from hqpo.evaluation import report as rp

bench = ROOT / "runs" / EXPERIMENT / "benchmark"
results = pd.read_csv(bench / "results.csv")
rp.benchmark_table(results)

In [ ]:
rp.fig_gap(results).show()
rp.fig_time(results).show()
rp.fig_qubits(pd.read_csv(bench / "qubit_growth.csv")).show()

In [ ]:
for fig in rp.fig_penalty(pd.read_csv(bench / "penalty_sweep.csv")):
    fig.show()
rp.fig_frontier(pd.read_csv(bench / "frontier.csv")).show()

## 5. Backtest runs

Synthetic data: this shows the pipeline end to end, not profitability.

In [ ]:
runs = rp.load_runs(ROOT / "runs")[EXPERIMENT]["backtests"]
rp.fig_equity(runs).show()
rp.backtest_table(runs, "test")